In [1]:
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from data_process import Date_Process
import json


## 数据处理

In [2]:

file_path = "./data/train.jsonl"
tokenizer_name = "pkuseg"
userdict_path = "./userdict/userdict.txt"
stopwords_path = ""
cleaning_parameters = [True, True, True, True]


In [3]:

data_p = Date_Process(file_path, tokenizer_name, userdict_path, stopwords_path, cleaning_parameters)

In [4]:
data_p.init()
print(len(data_p.train_data))


40000


## 特征表示

In [5]:

max_df = 0.8
min_df = 3
ngram_range = 1,3


In [6]:

tfidf_vec = TfidfVectorizer(max_df=max_df, min_df=min_df, ngram_range=tuple(ngram_range))
all_texts = [" ".join(i[0]) for i in data_p.all_data]
tfidf_matrix = tfidf_vec.fit_transform(all_texts)


## 模型训练


In [7]:

classifier_name = "SGD"


In [8]:

if classifier_name == "SGD":
    classifier = SGDClassifier()
elif classifier_name == "SVC":
    classifier = SVC()
elif classifier_name == "GBDT":
    classifier = GradientBoostingClassifier()
else:
    # 默认SGD
    classifier = SGDClassifier()


In [9]:

train_data = [" ".join(i[0]) for i in data_p.train_data]

train_tfidf_data = tfidf_vec.transform(train_data)

train_tfidf_tag = [data_p.tag2id[i[1]] for i in data_p.train_data]


In [10]:
from sklearn import set_config
set_config(display='text') 
classifier.fit(train_tfidf_data, train_tfidf_tag)


SGDClassifier()

In [11]:
# 模型测试
test_data = [" ".join(i[0]) for i in data_p.test_data]
test_tfidf_data = tfidf_vec.transform(test_data)
test_tfidf_tag = [data_p.tag2id[i[1]] for i in data_p.test_data]
predicted = classifier.predict(test_tfidf_data)


In [12]:

all_list = np.concatenate((test_tfidf_tag, predicted), axis=0)
all_list = np.unique(all_list)
target_names = [data_p.id2tag[i] for i in all_list]
acc = accuracy_score(test_tfidf_tag, predicted)
f1 = classification_report(test_tfidf_tag, predicted, target_names=target_names)
print('accuracy: {:.4}'.format(acc))
print(f1)


accuracy: 0.524
               precision    recall  f1-score   support

          edu       0.55      0.60      0.58       619
      finance       0.48      0.44      0.46       976
        house       0.54      0.53      0.53       367
       travel       0.48      0.40      0.44       690
         tech       0.51      0.50      0.50      1142
       sports       0.65      0.67      0.66       744
         game       0.60      0.64      0.62       582
      culture       0.54      0.45      0.49       779
          car       0.56      0.65      0.60       789
        story       0.42      0.27      0.33       210
entertainment       0.50      0.60      0.55       940
     military       0.50      0.43      0.46       700
  agriculture       0.51      0.49      0.50       546
        world       0.47      0.53      0.49       881
        stock       0.42      0.14      0.21        35

     accuracy                           0.52     10000
    macro avg       0.51      0.49      0.49   

## 单句预测

In [24]:
with open("./data/test.jsonl", "r", encoding="utf-8") as f:
    lines = f.readlines()
    test_data = []
    for line in lines:
        line = json.loads(line)
        test_data.append(line["sentence"])


In [25]:
test_data

['A股：2个细分领域龙头个股值得股民关注',
 '买套房不香吗？为什么会有人愿花600万买部手机？',
 '1000多家房地产公司转行养猪？你看到的仅仅是表面',
 '【指数短线破位后的思路梳理】',
 '在“慢”的过程中积攒“快”的爆发力',
 '出差32次、飞行40704公里，我看到了中国最真实的贫困',
 '三季度GDP增长4.9%，前三季度经济增长由负转正',
 '邓华：改变毛泽东军令打天津，识破美军仁川登陆，彭德怀赞好帮手',
 '号称今年最后一片银杏叶？大雪节气的高岭宿集，古村尽带黄金甲',
 '潮汕人经商有什么秘诀？',
 '从航母上跳水危险吗？觉得和从泳池跳板上跳水一样，就大错特错了',
 '请把手放到桌子底下（12月10日）',
 '恼人的雪（原创诗歌）',
 '真香！雪域孤岛边防连吃上火锅，表情亮了',
 '张一山《鹿鼎记》完成救赎，韦小宝崛起了，建宁实力演绎',
 '《时代周刊》给“2020”画了“红X”，史上仅出现过四次',
 '中国饭店协会副会长林聪：提升单体酒店经营能力至关重要',
 '长安凯程F70 征服高山雨林极限穿越试驾之旅',
 '皓影锐混动试驾：油耗低、空间大、配置高、底盘巨舒适',
 '13个月内四次减持！完美世界董事长套现25.87亿',
 '李叔同的《送别》：美国生，日本长，却在中国成为永远的经典',
 '刘诗诗颜值分析|轮廓',
 '深秋时节新疆尉犁沙漠胡杨秋韵迷人',
 '这像不像当下的A股市场，每次你做着大涨梦（大盘将强势突破3500点）时，总是被（一）跌（惊）醒！#不许笑#',
 '未来的好公司有哪些？12家中国公司入选《财富》杂志“未来50强”企业榜单',
 '京城四美里，你最看好谁？',
 '小说：避孕药被换维生素，妻子怀孕大怒，丈夫却在角落偷笑',
 '中集集团：中集海洋渔业拥有两个专业化公司已成为我国深远海养殖的头部企业',
 '从《鹿鼎记》到《大秦赋》，曾迷倒意大利土豪的朱珠，有多牛？',
 '棉花 缺乏上涨驱动力',
 '安徽和县：打造长三角后花园',
 '股票投资，你的安全边际在哪里？',
 '老网红创新餐饮焕发新生',
 '新一轮5G建设招标开启',
 '在旅行的路上 依维柯欧胜房车让我找到家的感觉',
 '未来AI企业核心竞争力在哪？谁有应用标准，谁有“王牌”',
 '清凉寺墓地与运城盐湖',


In [26]:
import random
sample_5 = random.sample(test_data, 5)

In [27]:
sample_5

['格林美：公司管理层将持续坚守“城市矿山+新能源材料”核心产业战略',
 '高尔夫美女不解为什么没人愿意接近她，面对粉丝质疑，她正面回击：不要仅凭穿衣打扮，就认定我是轻浮的女性',
 '一生只为书法活',
 '兵哥哥找教师对象的理由',
 '为什么要读书？读什么书？胡适谈读书之惑']

In [28]:
words_list = []
for word in sample_5:
    word = data_p.data_cleaning(word, [True, True, True, True])
    seg_list = data_p.tokenizer.cut(word)
    seg_list = [i for i in seg_list if i != '' and i not in data_p.stopwords]
    words_list.append(seg_list)


In [29]:
words_list

[['格林美',
  ':',
  '公司',
  '管理层',
  '将',
  '持续',
  '坚守',
  '“',
  '城市',
  '矿山',
  '+',
  '新',
  '能源',
  '材料',
  '”',
  '核心',
  '产业',
  '战略'],
 ['高尔夫',
  '美女',
  '不解',
  '为什么',
  '没',
  '人',
  '愿意',
  '接近',
  '她',
  ',',
  '面对',
  '粉丝',
  '质疑',
  ',',
  '她',
  '正面',
  '回击',
  ':',
  '不要',
  '仅',
  '凭',
  '穿衣',
  '打扮',
  ',',
  '就',
  '认定',
  '我',
  '是',
  '轻浮',
  '的',
  '女性'],
 ['一生', '只', '为', '书法', '活'],
 ['兵哥哥', '找', '教师', '对象', '的', '理由'],
 ['为什么', '要', '读书', '?', '读', '什么', '书', '?', '胡适', '谈', '读书', '之', '惑']]

In [36]:
test_data = [" ".join(i) for i in words_list]
test_tfidf_data = tfidf_vec.transform(test_data)
predicted = classifier.predict(test_tfidf_data)
print({data_p.id2tag[predicted[n]]: sample_5[n] for n in range(len(predicted))})


{'finance': '格林美：公司管理层将持续坚守“城市矿山+新能源材料”核心产业战略', 'entertainment': '高尔夫美女不解为什么没人愿意接近她，面对粉丝质疑，她正面回击：不要仅凭穿衣打扮，就认定我是轻浮的女性', 'culture': '一生只为书法活', 'edu': '为什么要读书？读什么书？胡适谈读书之惑'}
